In [2]:
import numpy as np

def optimize_bid_1():
    fair_value = 920
    min_reserve = 670
    max_reserve = 920
    step = 5

    # Generate all possible reserve prices (R) based on the problem description
    possible_reserves = np.arange(min_reserve, max_reserve + step, step)
    num_scenarios = len(possible_reserves)

    best_bid = 0
    max_ev = -float('inf')
    results = []

    # Iterate through all valid bid increments
    for bid in np.arange(min_reserve, max_reserve + step, step):
        # A trade occurs if bid >= R
        # In a uniform distribution with discrete steps, 
        # P(trade) = (Number of R <= bid) / (Total number of possible R)
        trades_won = np.sum(bid > possible_reserves)
        prob_trade = trades_won / num_scenarios
        
        profit_per_trade = fair_value - bid
        expected_value = prob_trade * profit_per_trade
        
        results.append((bid, prob_trade, expected_value))
        
        if expected_value > max_ev:
            max_ev = expected_value
            best_bid = bid

    # Display results
    print(f"{'Bid':<10} | {'Prob(Trade)':<15} | {'Expected PnL':<15}")
    print("-" * 45)
    for b, p, ev in results:
        # Highlighting the optimal range
        marker = " <--- OPTIMAL" if b == best_bid else ""
        print(f"{b:<10} | {p:<15.2%} | {ev:<15.2f}{marker}")

    return best_bid

if __name__ == "__main__":
    x = optimize_bid_1()
    print(x)

Bid        | Prob(Trade)     | Expected PnL   
---------------------------------------------
670        | 0.00%           | 0.00           
675        | 1.96%           | 4.80           
680        | 3.92%           | 9.41           
685        | 5.88%           | 13.82          
690        | 7.84%           | 18.04          
695        | 9.80%           | 22.06          
700        | 11.76%          | 25.88          
705        | 13.73%          | 29.51          
710        | 15.69%          | 32.94          
715        | 17.65%          | 36.18          
720        | 19.61%          | 39.22          
725        | 21.57%          | 42.06          
730        | 23.53%          | 44.71          
735        | 25.49%          | 47.16          
740        | 27.45%          | 49.41          
745        | 29.41%          | 51.47          
750        | 31.37%          | 53.33          
755        | 33.33%          | 55.00          
760        | 35.29%          | 56.47          
765        | 3

In [3]:
import numpy as np
import pandas as pd
from itertools import product

FAIR_VALUE = 920
POSSIBLE_RESERVES = np.arange(670, 925, 5)

DISTRIBUTIONS = [
    "rational_795", "rational_800", "rational_810", "rational_820",
    "rational_tight", "rational_wide", "rational_laplace",
    "rational_higher", "rational_lower",
    "bimodal", "uniform_rational", "conservative", "aggressive"
]

N_OPPONENTS_LIST = [3000, 3250, 3500]
N_COUNTERPARTIES_LIST = [30, 50, 75, 100]
B2_VALUES = np.arange(795, 921, 5)

N_TRIALS = 100_000
BATCH_SIZE = 10_000

rng = np.random.default_rng(42)


def generate_rational_sum(dist_name, n_samples, n_rational):
    """
    Generate only the SUM of rational players' b2 values for a batch.
    This avoids storing unnecessary full opponent matrices for longer than needed.
    """

    if dist_name == "rational_795":
        x = rng.normal(795, 30, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_800":
        x = rng.normal(800, 30, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_810":
        x = rng.normal(810, 30, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_820":
        x = rng.normal(820, 30, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_tight":
        x = rng.normal(800, 15, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_wide":
        x = rng.normal(800, 50, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_laplace":
        x = rng.laplace(800, 20, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_higher":
        x = rng.normal(840, 30, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "rational_lower":
        x = rng.normal(760, 30, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "bimodal":
        half = n_rational // 2
        x1 = rng.normal(780, 20, (n_samples, half))
        x2 = rng.normal(820, 20, (n_samples, n_rational - half))
        x = np.concatenate([x1, x2], axis=1)
        x = np.clip(x, 670, 920)

    elif dist_name == "uniform_rational":
        x = rng.uniform(670, 920, (n_samples, n_rational))

    elif dist_name == "conservative":
        x = rng.normal(800, 10, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    elif dist_name == "aggressive":
        x = rng.normal(860, 30, (n_samples, n_rational))
        x = np.clip(x, 670, 920)

    else:
        raise ValueError(f"Unknown distribution: {dist_name}")

    return x.sum(axis=1)


def run_one_scenario(dist_name, n_opponents, n_counterparties, b2_values, n_trials, batch_size):
    """
    Runs one scenario in batches.

    Instead of creating:
        1_000_000 x 3000 opponent matrix

    we create:
        20_000 x 3000 matrix per batch

    Much safer for RAM.
    """

    n_random = 500
    n_rational = n_opponents - n_random

    pnl_store = {b2: [] for b2 in b2_values}

    for start in range(0, n_trials, batch_size):
        m = min(batch_size, n_trials - start)

        # Opponent average b2 for this batch
        rational_sum = generate_rational_sum(dist_name, m, n_rational)
        random_sum = rng.uniform(670, 920, (m, n_random)).sum(axis=1)

        opponent_avg_b2 = (rational_sum + random_sum) / n_opponents

        # Reserve prices for available counterparties
        reserves = rng.choice(
            POSSIBLE_RESERVES,
            size=(m, n_counterparties),
            replace=True
        )

        for b2 in b2_values:
            if b2 == FAIR_VALUE:
                pnls = np.zeros(m)
            else:
                trades = (reserves < b2).sum(axis=1)
                profit_per_trade = FAIR_VALUE - b2

                # Correct version:
                # include your bid in the average among all opponents + you
                full_avg = (opponent_avg_b2 * n_opponents + b2) / (n_opponents + 1)

                above_avg = b2 > full_avg

                penalty = np.where(
                    above_avg,
                    1.0,
                    ((FAIR_VALUE - full_avg) / (FAIR_VALUE - b2)) ** 3
                )

                pnls = trades * profit_per_trade * penalty

            pnl_store[b2].append(pnls)

    results = []

    for b2 in b2_values:
        all_pnls = np.concatenate(pnl_store[b2])

        results.append({
            "b2": b2,
            "mean": round(all_pnls.mean(), 2),
            "std": round(all_pnls.std(), 2),
            "p10": round(np.percentile(all_pnls, 10), 2),
            "p25": round(np.percentile(all_pnls, 25), 2),
        })

    return results


rows = []
total = len(DISTRIBUTIONS) * len(N_OPPONENTS_LIST) * len(N_COUNTERPARTIES_LIST)
done = 0

for dist, n_opp, n_cp in product(DISTRIBUTIONS, N_OPPONENTS_LIST, N_COUNTERPARTIES_LIST):
    print(f"Running: {dist}, n_opp={n_opp}, n_cp={n_cp}...")

    results = run_one_scenario(
        dist_name=dist,
        n_opponents=n_opp,
        n_counterparties=n_cp,
        b2_values=B2_VALUES,
        n_trials=N_TRIALS,
        batch_size=BATCH_SIZE
    )

    for r in results:
        rows.append({
            "dist": dist,
            "n_opponents": n_opp,
            "n_counterparties": n_cp,
            **r
        })

    done += 1
    print(f"  {done}/{total} done")


df = pd.DataFrame(rows)

print("\n── best b2 per scenario ──────────────────────────────────────────")
best = df.loc[df.groupby(["dist", "n_opponents", "n_counterparties"])["mean"].idxmax()]
print(best[["dist", "n_opponents", "n_counterparties", "b2", "mean", "p10"]].to_string(index=False))

print("\n── how often each b2 is optimal ──────────────────────────────────")
print(best["b2"].value_counts().sort_index())

print("\n── overall best b2 by mean across all scenarios ──────────────────")
print(df.groupby("b2")["mean"].mean().sort_values(ascending=False).head(10))

# Optional save
df.to_csv("b2_sweep_results.csv", index=False)
best.to_csv("b2_best_per_scenario.csv", index=False)

Running: rational_795, n_opp=3000, n_cp=30...
  1/156 done
Running: rational_795, n_opp=3000, n_cp=50...
  2/156 done
Running: rational_795, n_opp=3000, n_cp=75...
  3/156 done
Running: rational_795, n_opp=3000, n_cp=100...
  4/156 done
Running: rational_795, n_opp=3250, n_cp=30...
  5/156 done
Running: rational_795, n_opp=3250, n_cp=50...
  6/156 done
Running: rational_795, n_opp=3250, n_cp=75...
  7/156 done
Running: rational_795, n_opp=3250, n_cp=100...
  8/156 done
Running: rational_795, n_opp=3500, n_cp=30...
  9/156 done
Running: rational_795, n_opp=3500, n_cp=50...
  10/156 done
Running: rational_795, n_opp=3500, n_cp=75...
  11/156 done
Running: rational_795, n_opp=3500, n_cp=100...
  12/156 done
Running: rational_800, n_opp=3000, n_cp=30...
  13/156 done
Running: rational_800, n_opp=3000, n_cp=50...
  14/156 done
Running: rational_800, n_opp=3000, n_cp=75...
  15/156 done
Running: rational_800, n_opp=3000, n_cp=100...
  16/156 done
Running: rational_800, n_opp=3250, n_cp=30...